In [41]:
import pandas as pd

In [42]:
df = pd.read_csv('../data/churn_cleaned.csv')

In [43]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [44]:
#transform the target variable into a dummy variable
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [45]:
#verification that it worked
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [46]:
# Verify tenure=0 exists
print(df[df['tenure'] == 0].shape[0])  # Count of customers with tenure 0

# See actual values
print(df['tenure'].value_counts().sort_index().head(10))

11
tenure
0     11
1    613
2    238
3    200
4    176
5    133
6    110
7    131
8    123
9    119
Name: count, dtype: int64


In [47]:
#create tenure groups 
mybins = [-1, 0, 12, 24, 36, df['tenure'].max()]
mylabels = ['New', '0-1 year', '1-2 years', '2-3 years', '3+ years']

df['tenure_groups'] = pd.cut(x= df['tenure'], bins= mybins, labels= mylabels, include_lowest=True)

The purpose of this step is to feature split a vague column into more interpretable data:
- After some research, telecommunication company typically have contract ranging from 1 to 3 years.
- Therefore, 5 groups were created that represents:
    - 'New' = Brand new customers (0 months)
    - '0-1 year' = First year customers (1-12 months)
    - '1-2 years' = Second year (13-24 months)
    - '2-3 years' = Third year (25-36 months)
    - '3+ years' = Super loyal (37+ months)

- This makes the data more understandable.
- It will also help the model performance.

In [48]:
#identify categorical and numerical columns
categorical = df.select_dtypes(include= ['object', 'category']).columns
numerical = df.select_dtypes(include= ['float', 'int']).columns

print(f"Categorical columns: \n{categorical}\n")
print(f"Numerical columns: \n{numerical}")

Categorical columns: 
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'tenure_groups'],
      dtype='object')

Numerical columns: 
Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn'], dtype='object')


Issue encountered:
 
When scaling and training the trainig data, got an error as the tenure_group column was neither a int, float or object dtype but a category dtype. So I had to add 'category' for categorical columns for it to work.

In [49]:
#Transform all categorical variables to dummy variables
df_ready = pd.get_dummies(data= df, columns= categorical, drop_first= True, dtype= int)

In [50]:
from sklearn.model_selection import train_test_split
X = df_ready.drop(columns= ['Churn', 'tenure'])
y = df_ready['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify= y)

The stratify hyperparameter allows for the train and test data to keep the same distribution. Since the target variable is not balacnced (73% = No, 27% = Yes) this is crucial for evaluation accuracy.

In [51]:
#initialize scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [52]:
#fit numerical training data only and scale training data and test data
cols_to_scale = ['MonthlyCharges', 'TotalCharges']

# Create a copy of the original data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Scale ONLY the specified columns
scaler = StandardScaler()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [53]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)

X_test_scaled_df = pd.DataFrame(X_test_scaled, columns= X_test.columns, index=X_test.index)

In [ ]:
#save each
import joblib

X_train_scaled_df.to_csv('../data/X_train.csv', index=False)
X_test_scaled_df.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)
joblib.dump(scaler, '../models/scaler.pkl') #needed for the prediction script

['../models/scaler.pkl']

In this step, feature engineering was performed to transform the cleaned dataset into a model-ready format. The target variable was encoded, tenure-based features were created, categorical variables were one-hot encoded, and the dataset was split into training and testing sets. Numerical features were standardized to ensure compatibility with machine learning models.
